<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/RAG_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG 101: Why Models Need Your Data in the Prompt

Every LLM has blind spots it cannot think its way out of: **your private data** (internal docs, product specs, support tickets it was never trained on), **fresh events** (anything after its training cutoff), and **your application's specifics** (what *your* system actually does). Retrieval-Augmented Generation (RAG) fixes all three the same way: find the right text, put it in the prompt, and let the model answer *from it*.

This notebook shows the core move of RAG in its smallest possible form — no embeddings, no vector database yet. Just: a question the model cannot answer, a document that answers it, and a prompt that brings them together.

## 🧭 What You'll Learn

- The three permanent blind spots of every LLM: private data, post-cutoff events, and your app's internals
- The two failure modes when a model is asked about them: honest refusal and **confident fabrication**
- The core RAG move: *augmenting* the prompt with retrieved context (here, "retrieval" is us pasting a document)
- The course-standard context-boundary prompt template you will reuse in every RAG notebook
- Why context isn't free — and why that forces us to build real retrieval next

## 1. Setup: Environment, Keys, and Provider

The standard course setup cell — pick your provider, and it handles installs (Colab) and API keys (Colab Secrets or a local `.env`).

The **model field is an editable dropdown** (`{allow-input: true}`): pick one of the listed course defaults, or type any newer model ID straight into the box — no code changes needed. (Locally, simply edit the string.)

In [1]:
# ============================================================
# ⚙️ Setup — environment, dependencies, API keys, provider
# ============================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Pick your model provider (dropdown in Colab; edit the value locally)
PROVIDER = "gemini"  # @param ["gemini", "openai", "anthropic"]

# Pick a model for the selected provider — or TYPE any newer model ID into the
# box (the dropdown is editable thanks to allow-input):
CHAT_MODEL = "gemini-3.7-flash"  # @param ["gemini-3.7-flash", "gemini-3.5-flash-lite", "gpt-5.6-luna", "claude-sonnet-5"] {allow-input: true}

REQUIRED_KEYS = {
    "gemini": ["GOOGLE_API_KEY"],
    "openai": ["OPENAI_API_KEY"],
    "anthropic": ["ANTHROPIC_API_KEY"],
}[PROVIDER]  # only the selected provider's key is required

if IN_COLAB:
    import importlib
    import site
    import subprocess

    # Shared install profile, pinned course-wide (July 2026). Library updates can
    # change behavior, so we pin versions to keep every cell reproducible.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "google-genai==2.3.0",
            "openai==2.46.0",
            "anthropic==0.117.0",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without a runtime restart

    # In Colab: Secrets tab (🔑 icon in the left sidebar) → Add new secret →
    # name it e.g. GOOGLE_API_KEY, paste the key, and toggle notebook access on.
    from google.colab import userdata

    for key in REQUIRED_KEYS:
        os.environ[key] = userdata.get(key)

if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    # Locally: install dependencies once from the repo's requirements file.
    # API keys live in a .env file at the repo root (never hardcode keys in cells).
    from dotenv import load_dotenv

    load_dotenv()
    missing = [k for k in REQUIRED_KEYS if not os.getenv(k)]
    assert not missing, f"Missing from .env: {missing}"

print(f"✅ Setup complete — {'Colab' if IN_COLAB else 'local'} | provider: {PROVIDER}")

✅ Setup complete — local | provider: gemini


## 2. Clients and the `generate()` Helper

📎 *Built in the "How To Use LLMs via API" notebook — one function, three SDK branches, selected by `PROVIDER`.*

In [2]:
# 📎 This pattern was built in the "How To Use LLMs via API" notebook.
from anthropic import Anthropic
from google import genai
from google.genai import types as genai_types
from openai import OpenAI

# Course-standard default models per provider (August 2026)
MODELS = {
    "gemini": "gemini-3.7-flash",
    "openai": "gpt-5.6-luna",
    "anthropic": "claude-sonnet-5",
}

# The setup-cell form selection (or any typed model ID) overrides the default:
MODELS[PROVIDER] = CHAT_MODEL

# Create the client for the selected provider (only its key is required)
if PROVIDER == "gemini":
    gemini_client = genai.Client()
elif PROVIDER == "openai":
    openai_client = OpenAI()
elif PROVIDER == "anthropic":
    anthropic_client = Anthropic()


def generate(prompt, system=None, model=None):
    """Send one prompt to the selected PROVIDER and return the reply text."""
    if PROVIDER == "gemini":
        response = gemini_client.models.generate_content(
            model=model or MODELS["gemini"],
            contents=prompt,
            config=genai_types.GenerateContentConfig(system_instruction=system),
        )
        return response.text

    if PROVIDER == "openai":
        response = openai_client.responses.create(
            model=model or MODELS["openai"],
            instructions=system,
            input=prompt,
            reasoning={"effort": "none"},
        )
        return response.output_text

    if PROVIDER == "anthropic":
        response = anthropic_client.messages.create(
            model=model or MODELS["anthropic"],
            max_tokens=4096,
            # Anthropic rejects system=None, so only pass it when set
            **({"system": system} if system else {}),
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")

## 3. A Question the Model Cannot Answer

We need a question whose answer no public model was ever trained on. Conveniently, we have one: the internal build notes of *this course's own production AI Tutor*. The details below are private project documentation — a model can only know them if **we** put them in the prompt.

That is the honest shape of the problem RAG solves. (Old versions of this demo asked about a just-released model — e.g. "how many parameters does Llama 4 have?" — but that breaks as soon as newer models learn the answer. A private fact stays private forever, so this demo cannot rot.)

First, ask *without* giving the model the document:

In [3]:
QUESTION = (
    "In the AI Tutor project's retrieval stack, what chunk size and overlap does the "
    "heading-aware chunker use, and how are the dense and lexical result lists combined?"
)

# No context provided — the model is on its own.
response = generate(
    "Be concise and take your time to answer the following question.\n"
    f"Question: {QUESTION}\nAnswer:",
    system="You are an assistant and expert in answering questions.",
)
print(response)

In the AI Tutor project's retrieval stack:

* **Chunk Size & Overlap:** The heading-aware chunker uses a target chunk size of **500 to 1,000 tokens/characters** (commonly configured to **1,000 characters with a 100–200 character overlap** or **512 tokens with a 50–100 token overlap**, depending on the document structure) while preserving semantic sections under markdown headings.
* **Combining Dense & Lexical Results:** Dense (vector semantic search) and lexical (BM25 keyword search) result lists are merged using **Reciprocal Rank Fusion (RRF)** (and optionally re-scored with a cross-encoder reranker) to produce a unified, balanced relevance ranking.


**What just happened?** One of two things — and both are instructive. Either the model admitted it doesn't know this project, or (worse) it produced a *plausible-sounding* answer: some reasonable chunk size, some standard fusion method. Reasonable-sounding is the dangerous case — nothing in the reply signals "I made this up". **Your output may differ from ours; the failure mode is the point, not the exact wording.**

## 4. Augmenting the Prompt

Now the RAG move. We take the document that actually contains the answer and place it in the prompt, inside explicit context boundaries. The template below — rules in the system prompt, context between `<START_OF_CONTEXT>` tags, then the question — is the **course-standard augmentation template**; you will meet it again in every RAG notebook, growing more capable each time.

📎 *The boundary tags double as the trusted/untrusted separation from the prompt-injection lesson: retrieved text is data, never instructions.*

In [4]:
# Our "retrieved" document — for now, retrieval is us pasting it in.
# (These are the AI Tutor's real internal build notes: you will build this
# exact pipeline, piece by piece, over the coming lessons.)
ARTICLE = """AI TUTOR — INTERNAL RELEASE NOTES (build 2026.07, codename "Cobalt")

Retrieval stack
- The chunker is heading-aware: it splits documents at Markdown headings into chunks of
  at most 800 tokens with a 100-token overlap, and it never splits a code block in half.
- Dense retrieval uses Gemini embeddings; lexical retrieval uses BM25. The two ranked
  lists are merged with Reciprocal Rank Fusion (RRF) before reranking.
- The fused candidates are reranked with a Cohere reranker; the final prompt receives
  only the top-scoring chunks.

Answering
- Every answer must cite the source chunks it used. If retrieval confidence is low, the
  tutor answers "I don't know" instead of guessing.
- Answer generation runs behind a single generate() helper, so any of the three major
  model providers can serve as the answering model per deployment."""

SYSTEM_INSTRUCTION = (
    "You are an assistant and expert in answering questions from a chunks of content. "
    "Only answer AI-related question, else say that you cannot answer this question."
)

# Course-standard augmentation template: context inside explicit boundary tags.
PROMPT_TEMPLATE = (
    "Read the following informations that might contain the context you require to "
    "answer the question. You can use the informations starting from the "
    "<START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag. Here is the content:\n\n"
    "<START_OF_CONTEXT>\n{context}\n<END_OF_CONTEXT>\n\n"
    "Please provide an informative and accurate answer to the following question based "
    "on the available context. Be concise and take your time.\nQuestion: {question}\nAnswer:"
)

formatted_prompt = PROMPT_TEMPLATE.format(context=ARTICLE, question=QUESTION)

response = generate(formatted_prompt, system=SYSTEM_INSTRUCTION)
print(response)

Based on the provided context:

* **Chunk Size & Overlap:** The heading-aware chunker creates chunks of **at most 800 tokens** with a **100-token overlap** (without splitting code blocks in half).
* **Combination Method:** The dense and lexical ranked lists are merged using **Reciprocal Rank Fusion (RRF)** before being reranked.


**What just happened?** Same model, same question — but now the answer is specific and correct: 800-token chunks, 100-token overlap, BM25 + dense lists merged with Reciprocal Rank Fusion. The model didn't get smarter; its *prompt got better informed*. That is the entire trick of RAG: *retrieval* finds the right text, *augmentation* puts it in the prompt, *generation* answers from it. Today we did the retrieval step by hand.

In [5]:
# 🔬 OPTIONAL EXPERIMENT — Can the model refuse instead of fabricate?
# Ask something the document does NOT contain, with context still present.
# A well-behaved RAG answer should say the context doesn't cover it.
missing_question = "Which company hosts the AI Tutor's production database?"

print(generate(
    PROMPT_TEMPLATE.format(context=ARTICLE, question=missing_question),
    system=SYSTEM_INSTRUCTION,
))

Based on the provided context, there is no information mentioning which company hosts the AI Tutor's production database.


## 5. Context Isn't Free

Pasting documents into prompts works — so why not paste *everything you have* into every prompt? Because every word of context is tokens, and tokens are money and latency (and past a point, degraded attention). Check what our one small document costs:

In [6]:
words = len(ARTICLE.split(" "))
# Rough rule of thumb for English text: 1 token ≈ 0.75 words
approx_tokens = int(words / 0.75)
print(f"Context size: {words} words ≈ {approx_tokens} tokens — billed on EVERY question we ask.")

Context size: 132 words ≈ 176 tokens — billed on EVERY question we ask.


**What just happened?** One document is cheap. But a real knowledge base — the course's tutor indexes thousands of documentation pages — cannot be pasted wholesale into every prompt, even with today's million-token context windows. We need a system that *selects the few most relevant pieces* per question. Building that selector — embeddings, similarity search, top-k retrieval — is exactly the next lesson.

## 🔑 Key Takeaways

- LLMs permanently lack three things: your private data, post-cutoff events, and your application's internals. No model upgrade fixes that — only supplying the data does.
- Un-grounded models fail in two ways: refusal (fine) and **confident fabrication** (dangerous — it looks identical to a real answer).
- RAG = retrieval + augmentation + generation. The augmentation template with explicit context boundaries is course-standard from here on.
- Grounding also enables *refusal*: with context present, a model can honestly say "that's not in the document".
- Context is billed per question — you cannot paste your whole knowledge base, so you need retrieval that selects the right few chunks. That's the next lesson.